# Bài 8 · Xử lý dữ liệu thời gian

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu bài học** — sau notebook này, bạn sẽ:

1. Đưa cột ngày về kiểu ngày giờ (`datetime64`), khai thác `.dt.*` và tính khoảng thời gian (`Timedelta`).
2. Làm việc trên trục thời gian: cắt lát, tổng hợp theo kỳ (`resample`) và làm mượt bằng cửa sổ trượt (`rolling`).
3. So sánh theo thời gian: tỷ lệ thay đổi giữa hai kỳ (`pct_change`) và **cùng kỳ năm trước** (`shift(12)`).
4. Kiểm tra ba vấn đề: kỳ chưa trọn, định dạng ngày mơ hồ và ngày ghi nhận không phải ngày sự kiện.

Dữ liệu: **690.112 bản ghi đánh giá** của Santiago (bảng rút gọn hai cột) và bảng `listings`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

SNAPSHOT = pd.Timestamp("2026-06-29")
rv_raw = pd.read_csv(
    "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/reviews.csv",
    parse_dates=["date"],
)
so_dong_sau_moc = (rv_raw["date"] > SNAPSHOT).sum()
rv = rv_raw.loc[rv_raw["date"] <= SNAPSHOT].copy()
print(f"Dữ liệu gốc: {len(rv_raw):,} dòng; loại {so_dong_sau_moc} dòng sau mốc chụp dữ liệu.")
rv.head(3)

## 1. Kiểu ngày giờ (`datetime`) và các thuộc tính `.dt`

In [ ]:
# Chuỗi 03/07 có hai cách hiểu; khai báo format để không phải đoán
print(pd.to_datetime("03/07/2026", format="%m/%d/%Y"))  # 7 tháng 3
print(pd.to_datetime("03/07/2026", format="%d/%m/%Y"))  # 3 tháng 7

In [ ]:
# .dt.*: trích xuất các thành phần của ngày
rv["date"].dt.year.value_counts().sort_index().tail(5)

In [ ]:
# Ngày nào trong tuần có nhiều đánh giá được ghi nhận nhất? (0 = thứ Hai)
rv["date"].dt.dayofweek.value_counts().sort_index()

Số đánh giá được ghi nhận cao nhất vào Chủ nhật (6) và thứ Hai (0). Tuy nhiên, cột `date`
không cho biết ngày lưu trú hay ngày trả phòng, nên chưa thể kết luận nguyên nhân của mẫu hình này.

In [ ]:
# Khoảng thời gian (Timedelta): số năm kể từ đánh giá đầu tiên
li = pd.read_csv(
    "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/data/listings.csv.gz",
    usecols=["id", "first_review", "number_of_reviews"],
    parse_dates=["first_review"],
)
first = li["first_review"].where(li["first_review"] <= SNAPSHOT)
so_nam = (SNAPSHOT - first).dt.days / 365.25
so_nam.describe().round(1)

## 2. Trục thời gian: cắt lát, tổng hợp theo kỳ và cửa sổ trượt

In [ ]:
r = rv.set_index("date").sort_index()

# Cắt lát thời gian bằng một phần chuỗi ngày
print("Năm 2026    :", len(r.loc["2026"]), "đánh giá")
print("Tháng 3/2026:", len(r.loc["2026-03"]), "đánh giá")

In [ ]:
# Tổng hợp theo lịch bằng resample (ME = tháng, nhãn ở cuối tháng)
theo_thang = r.resample("ME").size()
theo_thang.tail(4)

File gốc có **204 dòng sau mốc chụp dữ liệu**: 155 dòng ngày 30/06 và 49 dòng ngày 01/07.
Sau khi loại các dòng này, tháng 6 vẫn là **kỳ chưa trọn** vì dữ liệu chỉ được chụp đến 29/06.
Tháng 5/2026 là tháng đầy đủ cuối cùng và sẽ được dùng cho các phép so sánh tiếp theo.

In [ ]:
thang_day_du = theo_thang.loc[:"2026-05-31"]

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(thang_day_du.index, thang_day_du.values, color="#999", lw=1, label="theo tháng")
ax.plot(thang_day_du.index, thang_day_du.rolling(6, center=True).mean(),
        color="#1E93AB", lw=2.5, label="trượt 6 tháng")
ax.legend(); ax.set_title("Số đánh giá theo tháng tại Santiago")
plt.tight_layout(); plt.show()

Chuỗi giảm rõ trong giai đoạn 2020–2021, trùng với thời kỳ COVID-19. Cửa sổ trượt (`rolling`) làm giảm
dao động ngắn hạn để xu hướng dài hơn dễ quan sát hơn; nó không giải thích nguyên nhân.

## 3. So sánh theo thời gian

### shift: dời dữ liệu để đối chiếu các kỳ

`shift(n)` đẩy mỗi giá trị xuống `n` dòng và sinh `NaN` ở đầu. Nhờ đó
`s / s.shift(1)` so mỗi kỳ với **kỳ liền trước**, còn `shift(12)` so với
**cùng kỳ năm trước** — đúng cái `pct_change` và phép so cùng kỳ bên dưới đang làm.

In [ ]:
s = pd.Series([10, 12, 15], index=["T1", "T2", "T3"])
s.shift(1)          # mỗi ô lấy giá trị của kỳ liền trước

In [ ]:
# So kỳ liền trước và cùng kỳ năm trước cho tháng đầy đủ cuối cùng
so_ky_truoc = thang_day_du.pct_change() * 100
so_cung_ky = (thang_day_du / thang_day_du.shift(12) - 1) * 100
pd.Series({
    "so với tháng 4/2026": so_ky_truoc.loc["2026-05-31"],
    "so với tháng 5/2025": so_cung_ky.loc["2026-05-31"],
}).round(1)

Tháng 5/2026 tăng **3%** so với tháng 4/2026 và tăng **53,7%** so với tháng 5/2025.
Hai phép so trả lời hai câu hỏi khác nhau; với dữ liệu có mùa vụ, so với cùng kỳ năm trước
thường là phép so phù hợp hơn để đánh giá thay đổi dài hạn.

In [ ]:
# Chỉ số mùa vụ: CHỈ tính trên các năm trọn vẹn (mỗi tháng góp mặt đủ 4 năm 2022–2025)
tron_nam = rv[(rv["date"] >= "2022-01-01") & (rv["date"] <= "2025-12-31")]
mua_vu = tron_nam.groupby(tron_nam["date"].dt.month).size()
(mua_vu / mua_vu.mean() * 100).round(0)

Chỉ số cao nhất ở **T7–T8 và T10–T11**, thấp nhất ở **T2**. Đây là mô tả mẫu hình,
không phải bằng chứng về nguyên nhân. T2 ít ngày hơn các tháng khác cũng ảnh hưởng tới số đếm.

Nếu gộp cả nửa đầu 2026, các tháng T1–T6 nhận thêm một năm dữ liệu còn T7–T12 thì không.
Vì vậy, chỉ số mùa vụ phải được tính trên các năm đầy đủ để mỗi tháng góp mặt cùng số lần.

## 4. Bài tập tại lớp

### Bài 1 — Tháng vàng của từng năm

Với mỗi năm 2022–2025, tìm **tháng có nhiều đánh giá nhất** (gợi ý: gom nhóm bằng `groupby` theo
`[dt.year, dt.month]` rồi dùng `idxmax` cho từng năm, hoặc tổng hợp bằng `resample("ME")` rồi `groupby(index.year)`).

In [ ]:
# TODO Bài 1:
thang_4nam = thang_day_du.loc["2022":"2025"]
dinh = thang_4nam.groupby(thang_4nam.index.year).idxmax()
dinh

### Bài 2 — Tuần hay tháng?

Tổng hợp theo **tuần** (`resample("W")`) cho giai đoạn 2025–2026, vẽ cùng một biểu đồ
với đường trượt 4 tuần. So với bản theo tháng: nhiễu hơn hay mịn hơn? Khi nào đáng
dùng tuần thay vì tháng?

In [ ]:
# TODO Bài 2:
theo_tuan = r.loc["2025":].resample("W").size().loc[:"2026-06-28"]
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(theo_tuan.index, theo_tuan.values, color="#bbb", lw=0.8)
ax.plot(theo_tuan.index, theo_tuan.rolling(4, center=True).mean(), color="#E8890C", lw=2)
ax.set_title("Theo tuần: nhiều chi tiết hơn — và nhiều nhiễu hơn")
plt.tight_layout(); plt.show()

### Bài 3 — Chỗ ở mới có đánh giá theo quý

Dùng bảng `li`: đếm số chỗ ở có `first_review` trong từng **quý** từ 2023 đến nay.
Bỏ quý chưa trọn cuối cùng rồi tìm quý có số lượng lớn nhất.

In [ ]:
# TODO Bài 3:
moi = li.loc[li["first_review"].le(SNAPSHOT)].dropna(subset=["first_review"]).set_index("first_review")
theo_quy = moi.loc["2023":].resample("QE").size().iloc[:-1]
print(theo_quy)
print("Quý có nhiều chỗ ở mới được đánh giá nhất:", theo_quy.idxmax().date())

## 5. Bài tự luyện — So sánh mùa vụ hai thành phố

Lấy thêm bảng đánh giá của **Rio de Janeiro**
(`https://data.insideairbnb.com/brazil/rj/rio-de-janeiro/2026-06-24/visualisations/reviews.csv`):

1. Tính chỉ số mùa vụ theo tháng (như mục 3) cho Rio — trên **các năm trọn 2022–2025**
   (bài học mục 3: mỗi tháng phải góp mặt đủ số năm như nhau).
2. Vẽ hai chỉ số Santiago và Rio trên cùng một hình (hai đường, 12 tháng).
3. Xác định tháng có chỉ số cao nhất của mỗi thành phố và so sánh vị trí hai đỉnh.
4. Viết 3 câu nhận xét — mỗi câu kèm con số.


In [ ]:
RUN_HOMEWORK = False

if RUN_HOMEWORK:
    rio = pd.read_csv(
        "https://data.insideairbnb.com/brazil/rj/rio-de-janeiro/"
        "2026-06-24/visualisations/reviews.csv", parse_dates=["date"])
    ...

---

## Tóm tắt bài học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| `parse_dates` và định dạng ngày rõ ràng; `.dt.*` trích thành phần ngày | Chuẩn bị đúng kiểu dữ liệu |
| Tổng hợp theo kỳ (`resample`); làm mượt bằng cửa sổ trượt (`rolling`) | Quan sát dữ liệu ở nhiều độ phân giải |
| Loại ngày sau mốc chụp dữ liệu và kỳ chưa trọn | Tránh so sánh các kỳ không tương đương |
| Dữ liệu mùa vụ cần so **cùng kỳ năm trước** (`shift(12)`) | Chọn mốc so sánh phù hợp |
| Ngày ghi nhận không phải ngày sự kiện; số đánh giá là biến đại diện | Giới hạn diễn giải |

**Bài tiếp theo:** ôn tập và thi giữa kỳ (🚫 đóng). Bài 10 tiếp tục với làm sạch dữ liệu có cấu trúc.